# 13 - Scoped Reset and Teardown

Resets validation state, removes only explicitly listed demo-owned Delta tables, or deletes only Fabric items recorded as created by this demo. Destructive item teardown is disabled by default, requires an exact confirmation phrase, runs in reverse dependency order, and never deletes a workspace or reused item.

In [ ]:
# PARAMETERS
operation = 'RESET_VALIDATION'  # RESET_VALIDATION | RESET_DATA | TEARDOWN_ITEMS
workspace_id = ''
dry_run = True
allow_destructive_teardown = False
confirmation = ''
preserve_deployment_audit = True

import re
from datetime import datetime, timezone

import requests

API_BASE = 'https://api.fabric.microsoft.com/v1'
RESULTS = []
VALID_OPERATIONS = {'RESET_VALIDATION','RESET_DATA','TEARDOWN_ITEMS'}
assert operation in VALID_OPERATIONS

VALIDATION_TABLES = [
    'validation_results','validation_idempotency_manifest','validation_results_production',
    'validation_idempotency_current','validation_idempotency_manifest_production','lineage_contract']
ENTERPRISE_TABLES = [
    'bronze_organization','bronze_route','bronze_aircraft_fleet','bronze_work_team','bronze_skill','bronze_shift',
    'bronze_employee','bronze_employee_skill','bronze_employee_roster','bronze_retail_outlet','bronze_retail_product',
    'bronze_flight_route','bronze_flight_leg','bronze_customer','bronze_passenger','bronze_booking',
    'bronze_boarding_event','bronze_baggage_journey','bronze_baggage_scan','bronze_ramp_service_task',
    'bronze_maintenance_work_order','bronze_retail_pos','bronze_turnaround_phase','bronze_customer_experience',
    'bronze_recommendation_event','bronze_event_quality_cases','silver_quarantine_events',
    'dim_organization','dim_route','dim_aircraft_fleet','dim_work_team','dim_skill','dim_shift','dim_employee',
    'bridge_employee_skill','dim_retail_outlet','dim_retail_product','dim_customer','dim_passenger',
    'dim_customer_segment','bridge_flight_route','fact_flight_leg','fact_employee_roster','fact_booking',
    'fact_boarding_event','fact_baggage_journey','fact_baggage_scan','fact_ramp_service_task',
    'fact_maintenance_work_order','fact_retail_pos','fact_turnaround_phase','fact_customer_experience','fact_recommendation',
    'gold_airline_route_performance','gold_baggage_performance','gold_workforce_coverage','gold_retail_performance',
    'gold_customer_experience','gold_turnaround_phase_performance','gold_persona_scorecard',
    'gold_data_agent_enterprise_context','gold_flight_operations_kpi','gold_passenger_flow_kpi','gold_baggage_kpi',
    'gold_workforce_kpi','gold_maintenance_kpi','gold_energy_sustainability_kpi','gold_commercial_kpi',
    'gold_incident_customer_kpi','gold_kpi_catalog']
GOLD_STAR_TABLES = [
    'gold_dim_date','gold_dim_time','gold_dim_airport','gold_dim_terminal','gold_dim_zone','gold_dim_gate',
    'gold_dim_stand','gold_dim_airline','gold_dim_aircraft_type','gold_dim_route','gold_dim_employee',
    'gold_dim_team','gold_dim_asset','gold_dim_retail_outlet','gold_dim_customer_segment','gold_fact_flight',
    'gold_fact_turnaround','gold_fact_turnaround_milestone','gold_fact_passenger_flow','gold_fact_queue',
    'gold_fact_baggage','gold_fact_roster','gold_fact_maintenance','gold_fact_asset_state','gold_fact_energy',
    'gold_fact_retail_transaction','gold_fact_incident','gold_fact_customer_experience','gold_fact_recommendation']
ENTERPRISE_TABLES = ENTERPRISE_TABLES + GOLD_STAR_TABLES
CORE_TABLES = [
    'agent_context','gold_it_service_health','gold_executive_scorecard','gold_spatial_operational_status',
    'gold_energy_efficiency','gold_asset_reliability','gold_gate_turnaround_performance','gold_terminal_flow_summary',
    'gold_airport_operational_health','gold_incidents_recent','gold_energy_summary','gold_gate_utilization',
    'gold_queue_by_hour','gold_turnaround_by_hour','gold_kpi_daily_summary','fact_zone_occupancy','fact_asset_state',
    'bridge_asset_location','bridge_gate_stand','dim_location','dim_asset','dim_stand','dim_checkpoint','dim_zone',
    'dim_terminal','dim_time','dim_date','fact_operational_incidents','fact_weather','fact_maintenance_events',
    'fact_energy_metering','fact_passenger_queue_metrics','fact_flight_turnaround_events','dim_runway_reference',
    'bridge_airport_group_assignment','bridge_airline_aircraft_eligibility','bridge_airline_airport_service',
    'dim_country','dim_airline','dim_service_team','dim_aircraft','dim_gate','dim_airport',
    'bronze_zone_occupancy','bronze_asset_state','bronze_twin_relationships','bronze_asset_registry',
    'bronze_stand_registry','bronze_checkpoint_registry','bronze_terminal_zones','bronze_operational_incidents',
    'bronze_weather','bronze_maintenance','bronze_energy','bronze_passenger_queue','bronze_flight_turnaround',
    'bronze_airline_airport_service','bronze_airline_aircraft_eligibility','bronze_runway_reference',
    'bronze_airport_group_assignment','bronze_operating_region','bronze_corporate_headquarters','bronze_country',
    'bronze_airline','bronze_service_team','bronze_aircraft','bronze_gate','bronze_airport','bronze_demo_config']


def record(name, artifact_type, status, detail=''):
    row={'operation':operation,'artifact_name':name,'artifact_type':artifact_type,'status':status,
         'detail':detail[:4000],'observed_at':datetime.now(timezone.utc),'is_synthetic':True}
    RESULTS.append(row)
    print(status,artifact_type,name,detail)
    return row


def drop_tables(table_names):
    for table_name in table_names:
        if dry_run:
            record(table_name,'DeltaTable','DRY_RUN','Would drop demo-owned table if it exists')
        else:
            spark.sql('DROP TABLE IF EXISTS `'+table_name.replace('`','')+'`')
            record(table_name,'DeltaTable','SUCCEEDED','Dropped if present')

In [ ]:
environment_name = globals().get('environment_name', 'dev')
resource_prefix = globals().get('resource_prefix', 'fao-demo')
expected_confirmation = f'DELETE {resource_prefix} {environment_name}'
assert environment_name in {'dev', 'test'}, 'Production reset and teardown are refused by this demonstration'
assert re.fullmatch(r'[a-z0-9-]{3,24}', resource_prefix), 'Invalid resource prefix'
assert resource_prefix == 'fao-demo', 'Unrecognized resource prefix; refusing destructive operation'
if operation in {'RESET_DATA', 'TEARDOWN_ITEMS'} and not dry_run:
    assert allow_destructive_teardown is True, 'allow_destructive_teardown must be explicitly enabled'
    assert confirmation == expected_confirmation, 'Exact environment-scoped confirmation token is required'
print('Guarded operation plan:', operation, 'environment=', environment_name, 'resource_prefix=', resource_prefix, 'dry_run=', dry_run)

if operation == 'RESET_VALIDATION':
    drop_tables(VALIDATION_TABLES)

elif operation == 'RESET_DATA':
    drop_tables(VALIDATION_TABLES + ENTERPRISE_TABLES + CORE_TABLES)

elif operation == 'TEARDOWN_ITEMS':
    if not dry_run:
        assert re.fullmatch(r'[0-9a-fA-F-]{36}', workspace_id), 'A runtime workspace GUID is required'
    allowed_names = {
        'Airport Operations Command Center', 'AirportOpsDataAgent', 'AirportOpsPersonaReports',
        'AirportOpsSharedModel', 'AirportOpsRealtime', 'AirportOpsEventhouse',
        'AirportOpsWarehouse', 'AirportOpsLakehouse',
    }
    dependency_rank = {
        'FabricApp': 8, 'DataAgent': 7, 'Report': 6, 'SemanticModel': 5,
        'KQLDatabase': 4, 'Eventhouse': 3, 'Warehouse': 2, 'Lakehouse': 1,
    }
    if not spark.catalog.tableExists('deployment_results'):
        record('deployment_results', 'DeploymentLedger', 'BLOCKED', 'No deployment ledger exists; no items are eligible for deletion')
        if not dry_run:
            raise RuntimeError('No deployment ledger exists')
    else:
        created_items = (
            spark.table('deployment_results')
            .filter((spark.table('deployment_results').deployment_status == 'SUCCEEDED') &
                    (spark.table('deployment_results').status_detail == 'Created item') &
                    (spark.table('deployment_results').item_id != ''))
            .select('artifact_name', 'artifact_type', 'item_id').distinct().collect()
        )
        eligible = [row for row in created_items if row['artifact_name'] in allowed_names and row['artifact_type'] in dependency_rank]
        eligible.sort(key=lambda row: dependency_rank[row['artifact_type']], reverse=True)
        for item in eligible:
            if dry_run:
                record(item['artifact_name'], item['artifact_type'], 'DRY_RUN', 'Would delete ledger-owned item ' + item['item_id'])
                continue
            response = requests.delete(
                API_BASE + '/workspaces/' + workspace_id + '/items/' + item['item_id'],
                headers={'Authorization': 'Bearer ' + notebookutils.credentials.getToken('pbi')}, timeout=90,
            )
            if response.status_code in {200, 204}:
                record(item['artifact_name'], item['artifact_type'], 'SUCCEEDED', 'Deleted ledger-owned item')
            elif response.status_code == 202 and response.headers.get('Location'):
                location = response.headers['Location']
                completed = False
                for _ in range(120):
                    status_response = requests.get(location, headers={'Authorization': 'Bearer ' + notebookutils.credentials.getToken('pbi')}, timeout=90)
                    body = status_response.json() if status_response.content else {}
                    status = str(body.get('status', '')).lower()
                    if status in {'succeeded', 'completed'}:
                        completed = True
                        break
                    if status in {'failed', 'cancelled'}:
                        break
                    __import__('time').sleep(int(status_response.headers.get('Retry-After', '3')))
                if completed:
                    record(item['artifact_name'], item['artifact_type'], 'SUCCEEDED', 'Deleted ledger-owned item after long-running operation')
                else:
                    record(item['artifact_name'], item['artifact_type'], 'FAILED', 'Delete operation did not report success')
            else:
                record(item['artifact_name'], item['artifact_type'], 'FAILED', response.text[:4000])
        if not eligible:
            record('Fabric items', 'DeploymentLedger', 'BLOCKED', 'No items were recorded as created by this demo')
        if not preserve_deployment_audit and not dry_run:
            spark.sql('DROP TABLE IF EXISTS deployment_results')
            record('deployment_results', 'DeploymentLedger', 'SUCCEEDED', 'Removed after item teardown by explicit configuration')

if RESULTS:
    try:
        spark.createDataFrame(RESULTS).write.mode('append').format('delta').saveAsTable('teardown_results')
    except Exception as exc:
        print('Teardown result log unavailable:', str(exc))

failures = [result for result in RESULTS if result['status'] == 'FAILED']
assert not failures, 'Reset/teardown had ' + str(len(failures)) + ' failures'
print(operation, 'complete with', len(RESULTS), 'recorded operations; dry_run=', dry_run)